# AIRO Tree-Based Models Example
## PhysioNet ICU Sepsis Prediction
### Created by Caden Wondell

We will compare:

1. Decision Tree
2. Random Forest
3. XGBoost

**Dataset:** Processed PhysioNet / Computing in Cardiology Challenge 2019 sepsis data

**Goal:** Predict the hourly `SepsisLabel` using ICU measurements available in the processed dataset.

The original PhysioNet challenge shifts the sepsis label six hours earlier, so the task is designed around early sepsis detection.

This notebook deliberately keeps all columns visible at first. We will inspect the dataset for possible leakage **before** deciding which columns should be excluded from the models.

## Step 1: Mount Google Drive

Mount Google Drive so the PhysioNet Parquet files can be loaded from the AIRO examples folder.

In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

## Step 2: Install the Required Packages

Install XGBoost for the boosting model and PyArrow so pandas can read Parquet files.

In [ ]:
%pip -q install xgboost pyarrow

## Step 3: Import the Libraries

Import the libraries needed to load the data, inspect possible leakage, split patients safely, train the three models, and compare their predictions.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

## Step 4: Set the Dataset File Paths

Both Parquet files are stored in the AIRO `Examples` folder.

- `physionet.parquet` contains the **hourly ICU measurements and SepsisLabel** used for modeling.
- `patient_summary.parquet` contains **patient-level metadata** about the original files and is not used as a predictor dataset.

In [ ]:
DATA_FOLDER = (
    "/content/drive/MyDrive/"
    "Colab_Notebooks/AIRO_CLUB/Examples/"
)

DATA_FILE = (
    DATA_FOLDER
    + "physionet.parquet"
)

SUMMARY_FILE = (
    DATA_FOLDER
    + "patient_summary.parquet"
)

print("Clinical dataset:")
print(DATA_FILE)
print()

print("Patient summary:")
print(SUMMARY_FILE)

## Step 5: Load the Clinical Dataset

Load `physionet.parquet`.

Each row represents an hourly ICU observation for a patient.

In [ ]:
data = pd.read_parquet(
    DATA_FILE
)

print(
    "Dataset shape:",
    data.shape
)

data.head()

## Step 6: Load the Patient Summary Metadata

Load `patient_summary.parquet` so we can inspect the patient-level metadata.

This file is useful for understanding the processed dataset, but its fields are **not used to train the models**.

In [ ]:
patient_summary = pd.read_parquet(
    SUMMARY_FILE
)

if "patient_id" not in patient_summary.columns:
    patient_summary = (
        patient_summary
        .reset_index()
    )

print(
    "Patient summary shape:",
    patient_summary.shape
)

patient_summary.head()

## Step 7: Look at All Clinical Dataset Columns

Do **not** remove anything yet.

First inspect every column so we can decide which variables are legitimate predictors and which may create leakage or shortcuts.

In [ ]:
print(
    "Dataset Shape:",
    data.shape
)

print()

print(
    "Columns:"
)

for column in data.columns:
    print(
        "-",
        column
    )

data.head()

## Step 8: Understand the Two Files

Compare the number of unique patients in the hourly clinical dataset with the patient summary file.

The modeling dataset should contain many rows per patient because each row represents a different ICU hour.

In [ ]:
clinical_patients = (
    data["patient_id"]
    .nunique()
)

summary_patients = (
    patient_summary["patient_id"]
    .nunique()
)

print(
    "Unique patients in physionet.parquet:",
    clinical_patients
)

print(
    "Unique patients in patient_summary.parquet:",
    summary_patients
)

print()

print(
    "Hourly clinical rows:",
    len(data)
)

print(
    "Average hourly rows per patient:",
    round(
        len(data) / clinical_patients,
        2
    )
)

## Step 9: Identify the Sepsis Target

The target supplied by the PhysioNet challenge is:

`SepsisLabel`

- **0 = No Sepsis label at this hour**
- **1 = Sepsis label at this hour**

The label is shifted earlier in the original challenge so that positive labels begin six hours before the defined sepsis onset.

In [ ]:
TARGET_COLUMN = "SepsisLabel"

print(
    "Target column:",
    TARGET_COLUMN
)

print()

print(
    "Unique target values:",
    sorted(
        data[TARGET_COLUMN]
        .dropna()
        .unique()
        .tolist()
    )
)

## Step 10: Examine Hourly Class Imbalance

Count how many hourly records have a SepsisLabel of 0 or 1.

Sepsis-positive hours are expected to be much less common, so accuracy alone may be misleading.

In [ ]:
hourly_class_counts = (
    data[TARGET_COLUMN]
    .value_counts()
    .sort_index()
)

hourly_class_percentages = (
    data[TARGET_COLUMN]
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

hourly_distribution = pd.DataFrame({
    "Count": hourly_class_counts,
    "Percent": hourly_class_percentages
})

hourly_distribution.index = [
    "No Sepsis",
    "Sepsis"
]

hourly_distribution

In [ ]:
hourly_class_counts.index = [
    "No Sepsis",
    "Sepsis"
]

hourly_class_counts.plot(
    kind="bar"
)

plt.title(
    "Hourly No Sepsis vs. Sepsis Labels"
)

plt.xlabel(
    "Class"
)

plt.ylabel(
    "Number of Hourly Records"
)

plt.xticks(
    rotation=0
)

plt.show()

## Step 11: Examine Sepsis at the Patient Level

A patient can have many hourly rows.

Count a patient as a sepsis patient if **any** of their hourly records has `SepsisLabel = 1`.

In [ ]:
patient_labels = (
    data
    .groupby("patient_id")[TARGET_COLUMN]
    .max()
)

patient_class_counts = (
    patient_labels
    .value_counts()
    .sort_index()
)

patient_class_percentages = (
    patient_labels
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

patient_distribution = pd.DataFrame({
    "Patients": patient_class_counts,
    "Percent": patient_class_percentages
})

patient_distribution.index = [
    "No Sepsis",
    "Sepsis"
]

patient_distribution

## Step 12: Check for Missing Values

Real ICU datasets contain many missing measurements because not every lab or vital sign is recorded every hour.

Display the columns with the highest percentage of missing values.

In [ ]:
missing_percent = (
    data
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
)

missing_table = pd.DataFrame({
    "Missing Percent": missing_percent
})

missing_table.head(20)

## Step 13: Review Possible Leakage Before Removing Columns

We still have **all original columns present**.

Now identify columns that need special attention before modeling.

### Direct target
`SepsisLabel` is the answer we are trying to predict and can never be included in `X`.

### Patient identifier
`patient_id` identifies the patient. It should be used to keep patients together during the train/test split, but not as a predictor.

### Dataset source
`source_set` identifies the source hospital/training set. It could let a model learn site-specific shortcuts rather than patient physiology.

### Time variables
`ICULOS` and `HospAdmTime` are official PhysioNet challenge variables and are available at prediction time, so they are **not automatically leakage**. We will leave them available and inspect how predictive they are.

In [ ]:
leakage_review = pd.DataFrame({
    "Column": [
        "SepsisLabel",
        "patient_id",
        "source_set",
        "ICULOS",
        "HospAdmTime"
    ],
    "Concern": [
        "Direct target",
        "Patient identifier",
        "Hospital/source shortcut",
        "Time variable - inspect, but available at prediction time",
        "Time variable - inspect, but available at prediction time"
    ],
    "Initial Decision": [
        "Exclude from predictors",
        "Exclude from predictors; use for grouped split",
        "Exclude from baseline model",
        "Keep unless analysis suggests a problem",
        "Keep unless analysis suggests a problem"
    ]
})

leakage_review

## Step 14: Check Whether Source Set Is Associated With Sepsis

`source_set` was added during processing to identify the source dataset.

If sepsis prevalence differs substantially by source, a model could use the source as a shortcut. We will inspect it before excluding it.

In [ ]:
source_sepsis = (
    data
    .groupby("source_set")[TARGET_COLUMN]
    .agg(
        [
            "count",
            "mean"
        ]
    )
    .rename(
        columns={
            "count": "Hourly Records",
            "mean": "Sepsis Rate"
        }
    )
)

source_sepsis["Sepsis Rate"] = (
    source_sepsis["Sepsis Rate"]
    * 100
)

source_sepsis

## Step 15: Create a Patient-Level Train/Test Split

This dataset has many rows from the same patient.

A normal random row split could put one patient's early ICU hours in training and the same patient's later hours in testing.

To prevent that leakage, split by `patient_id`.

**Every row from one patient must stay entirely in either training or testing.**

In [ ]:
group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    group_split.split(
        data,
        y=data[TARGET_COLUMN],
        groups=data["patient_id"]
    )
)

train_data = (
    data
    .iloc[train_index]
    .copy()
)

test_data = (
    data
    .iloc[test_index]
    .copy()
)

print(
    "Training rows:",
    len(train_data)
)

print(
    "Testing rows:",
    len(test_data)
)

print()

print(
    "Training patients:",
    train_data["patient_id"].nunique()
)

print(
    "Testing patients:",
    test_data["patient_id"].nunique()
)

## Step 16: Verify That No Patient Appears in Both Sets

This check should return **0 overlapping patients**.

In [ ]:
train_patients = set(
    train_data["patient_id"]
    .unique()
)

test_patients = set(
    test_data["patient_id"]
    .unique()
)

overlapping_patients = (
    train_patients
    .intersection(
        test_patients
    )
)

print(
    "Patients appearing in both sets:",
    len(overlapping_patients)
)

## Step 17: Screen Individual Features for Suspicious Predictive Power

Before deciding what to remove, test each numeric feature **by itself** using a one-split Decision Tree.

Because sepsis labels are highly imbalanced, raw accuracy is not a good leakage screen. Instead, we calculate **AUROC** from a patient-held-out test set.

A single feature with an unusually high AUROC deserves investigation. High predictive power does **not automatically mean leakage**—a legitimate clinical feature may simply be useful.

In [ ]:
candidate_numeric_features = [
    column
    for column in data.select_dtypes(
        include=np.number
    ).columns
    if column != TARGET_COLUMN
]

single_feature_results = []

for column in candidate_numeric_features:

    X_train_single = (
        train_data[
            [column]
        ]
    )

    y_train_single = (
        train_data[
            TARGET_COLUMN
        ]
    )

    X_test_single = (
        test_data[
            [column]
        ]
    )

    y_test_single = (
        test_data[
            TARGET_COLUMN
        ]
    )

    single_feature_model = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "classifier",
                DecisionTreeClassifier(
                    max_depth=1,
                    class_weight="balanced",
                    random_state=42
                )
            )
        ]
    )

    single_feature_model.fit(
        X_train_single,
        y_train_single
    )

    probabilities = (
        single_feature_model
        .predict_proba(
            X_test_single
        )[:, 1]
    )

    auc = roc_auc_score(
        y_test_single,
        probabilities
    )

    single_feature_results.append({
        "Feature": column,
        "AUROC": auc
    })

single_feature_results = (
    pd.DataFrame(
        single_feature_results
    )
    .sort_values(
        "AUROC",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

single_feature_results.head(15)

## Step 18: Visualize the Strongest Single-Feature Signals

This graph helps identify features that deserve closer review before modeling.

Do not automatically delete a feature simply because it is predictive.

In [ ]:
top_single_features = (
    single_feature_results
    .head(10)
    .sort_values(
        "AUROC"
    )
)

plt.figure(
    figsize=(9, 6)
)

plt.barh(
    top_single_features["Feature"],
    top_single_features["AUROC"]
)

plt.title(
    "Single-Feature Leakage / Shortcut Screen"
)

plt.xlabel(
    "AUROC Using One Feature and One Split"
)

plt.xlim(
    0.5,
    1.0
)

plt.show()

## Step 19: Choose the Predictor Columns After the Leakage Review

Only now do we decide which columns will be excluded from the model.

For this baseline:

- Remove `SepsisLabel` because it is the target.
- Remove `patient_id` because it is an identifier.
- Remove `source_set` because it identifies the source dataset/hospital and could create a site shortcut.
- Keep the original clinical measurements and official time variables.

No clinical feature is removed merely because it is strongly predictive.

In [ ]:
EXCLUDE_FROM_MODEL = [
    TARGET_COLUMN,
    "patient_id",
    "source_set"
]

MODEL_FEATURES = [
    column
    for column in data.columns
    if column not in EXCLUDE_FROM_MODEL
]

print(
    "Columns excluded from the models:"
)

for column in EXCLUDE_FROM_MODEL:
    print(
        "-",
        column
    )

print()

print(
    "Number of model features:",
    len(MODEL_FEATURES)
)

print()

print(
    "Model features:"
)

for column in MODEL_FEATURES:
    print(
        "-",
        column
    )

## Step 20: Create X and y for Training and Testing

Use the patient-level split that was already created.

This keeps the same patients out of both sets while using the selected clinical features.

In [ ]:
X_train = (
    train_data[
        MODEL_FEATURES
    ]
    .copy()
)

y_train = (
    train_data[
        TARGET_COLUMN
    ]
    .astype(int)
    .copy()
)

X_test = (
    test_data[
        MODEL_FEATURES
    ]
    .copy()
)

y_test = (
    test_data[
        TARGET_COLUMN
    ]
    .astype(int)
    .copy()
)

print(
    "X_train:",
    X_train.shape
)

print(
    "X_test:",
    X_test.shape
)

print()

print(
    "Training Sepsis Rate:",
    round(
        y_train.mean(),
        4
    )
)

print(
    "Testing Sepsis Rate:",
    round(
        y_test.mean(),
        4
    )
)

## Step 21: Prepare the Missing Values

Tree-based models do **not** require feature scaling.

We will fill missing numeric values using the median calculated from the training data.

The imputer is placed inside each model pipeline so the test set does not influence the imputation values.

In [ ]:
imputer = SimpleImputer(
    strategy="median"
)

print(
    "Number of numeric model features:",
    len(MODEL_FEATURES)
)

## Step 22: Calculate the Class Imbalance

Sepsis-positive hourly records are rare.

For the teaching example, we will give the minority class more weight during training so the models do not simply learn to predict "No Sepsis" most of the time.

Decision Tree and Random Forest use `class_weight`.

XGBoost uses `scale_pos_weight`.

In [ ]:
negative_count = int(
    (y_train == 0)
    .sum()
)

positive_count = int(
    (y_train == 1)
    .sum()
)

scale_pos_weight = (
    negative_count
    / positive_count
)

print(
    "No Sepsis training rows:",
    negative_count
)

print(
    "Sepsis training rows:",
    positive_count
)

print(
    "XGBoost scale_pos_weight:",
    round(
        scale_pos_weight,
        2
    )
)

# Decision Tree

## Step 23: Create the Decision Tree Model

Use a maximum depth of 4 so the tree stays interpretable.

`class_weight="balanced"` increases the importance of the less common Sepsis class during training.

In [ ]:
decision_tree = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=4,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

decision_tree

## Step 24: Train the Decision Tree

Train the model using only the training patients.

In [ ]:
decision_tree.fit(
    X_train,
    y_train
)

## Step 25: Make Decision Tree Predictions

Use the trained tree to classify the patient-hours in the untouched test set.

In [ ]:
dt_predictions = (
    decision_tree
    .predict(
        X_test
    )
)

dt_predictions[:10]

## Step 26: View the Decision Tree Confusion Matrix

Compare the Decision Tree predictions with the actual SepsisLabel values.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    dt_predictions,
    display_labels=[
        "No Sepsis",
        "Sepsis"
    ]
)

plt.title(
    "Decision Tree Confusion Matrix"
)

plt.show()

## Step 27: View the Decision Tree Classification Metrics

View precision, recall, F1 score, and overall accuracy.

In [ ]:
print(
    classification_report(
        y_test,
        dt_predictions,
        target_names=[
            "No Sepsis",
            "Sepsis"
        ],
        zero_division=0
    )
)

## Step 28: Save the Decision Tree Results

Save the positive-class metrics so they can be compared with Random Forest and XGBoost.

In [ ]:
dt_accuracy = accuracy_score(
    y_test,
    dt_predictions
)

dt_precision = precision_score(
    y_test,
    dt_predictions,
    zero_division=0
)

dt_recall = recall_score(
    y_test,
    dt_predictions,
    zero_division=0
)

dt_f1 = f1_score(
    y_test,
    dt_predictions,
    zero_division=0
)

print(
    "Accuracy:",
    round(
        dt_accuracy,
        3
    )
)

print(
    "Precision:",
    round(
        dt_precision,
        3
    )
)

print(
    "Recall:",
    round(
        dt_recall,
        3
    )
)

print(
    "F1:",
    round(
        dt_f1,
        3
    )
)

## Step 29: Visualize the Decision Tree

Display the actual rules learned from the PhysioNet ICU features.

The split thresholds shown in the tree are learned from the training patients.

In [ ]:
dt_classifier = (
    decision_tree
    .named_steps[
        "classifier"
    ]
)

plt.figure(
    figsize=(24, 12)
)

plot_tree(
    dt_classifier,
    feature_names=MODEL_FEATURES,
    class_names=[
        "No Sepsis",
        "Sepsis"
    ],
    filled=True,
    rounded=True,
    fontsize=8
)

plt.title(
    "Decision Tree for Sepsis Prediction"
)

plt.show()

# Random Forest

## Step 30: Create the Random Forest Model

Create a Random Forest containing 200 Decision Trees.

The forest uses bootstrap sampling and random subsets of features.

`class_weight="balanced_subsample"` adjusts the class weights inside each bootstrap sample.

In [ ]:
random_forest = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest

## Step 31: Train the Random Forest

Train all 200 trees using the same training patients used for the Decision Tree.

In [ ]:
random_forest.fit(
    X_train,
    y_train
)

## Step 32: Make Random Forest Predictions

Combine the trees' predictions to classify the same untouched test set.

In [ ]:
rf_predictions = (
    random_forest
    .predict(
        X_test
    )
)

rf_predictions[:10]

## Step 33: View the Random Forest Confusion Matrix

Compare the Random Forest predictions with the actual sepsis labels.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_predictions,
    display_labels=[
        "No Sepsis",
        "Sepsis"
    ]
)

plt.title(
    "Random Forest Confusion Matrix"
)

plt.show()

## Step 34: View the Random Forest Classification Metrics

View precision, recall, F1 score, and overall accuracy.

In [ ]:
print(
    classification_report(
        y_test,
        rf_predictions,
        target_names=[
            "No Sepsis",
            "Sepsis"
        ],
        zero_division=0
    )
)

## Step 35: Save the Random Forest Results

Save the metrics so they can be compared with the other models.

In [ ]:
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_precision = precision_score(
    y_test,
    rf_predictions,
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_predictions,
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_predictions,
    zero_division=0
)

print(
    "Accuracy:",
    round(
        rf_accuracy,
        3
    )
)

print(
    "Precision:",
    round(
        rf_precision,
        3
    )
)

print(
    "Recall:",
    round(
        rf_recall,
        3
    )
)

print(
    "F1:",
    round(
        rf_f1,
        3
    )
)

## Step 36: View Random Forest Feature Importance

Show the 10 features the Random Forest relied on most strongly.

Feature importance shows what the model used for prediction; it does **not** prove causation.

In [ ]:
rf_classifier = (
    random_forest
    .named_steps[
        "classifier"
    ]
)

rf_importance = pd.Series(
    rf_classifier.feature_importances_,
    index=MODEL_FEATURES
)

rf_importance = (
    rf_importance
    .sort_values(
        ascending=False
    )
    .head(10)
    .sort_values()
)

plt.figure(
    figsize=(9, 6)
)

plt.barh(
    rf_importance.index,
    rf_importance.values
)

plt.title(
    "Random Forest - Top 10 Feature Importances"
)

plt.xlabel(
    "Feature Importance"
)

plt.show()

# XGBoost

## Step 37: Create the XGBoost Model

Create an XGBoost classifier using shallow trees and a small learning rate.

Each boosting round adds another tree that improves the current model.

`scale_pos_weight` gives additional importance to the rare Sepsis class.

In [ ]:
xgboost_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "classifier",
            XGBClassifier(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.05,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgboost_model

## Step 38: Train XGBoost

Train XGBoost sequentially on the same training patients used for the other models.

In [ ]:
xgboost_model.fit(
    X_train,
    y_train
)

## Step 39: Make XGBoost Predictions

Use the trained XGBoost model to predict the same untouched test set.

In [ ]:
xgb_predictions = (
    xgboost_model
    .predict(
        X_test
    )
)

xgb_predictions[:10]

## Step 40: View the XGBoost Confusion Matrix

Compare the XGBoost predictions with the actual sepsis labels.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_predictions,
    display_labels=[
        "No Sepsis",
        "Sepsis"
    ]
)

plt.title(
    "XGBoost Confusion Matrix"
)

plt.show()

## Step 41: View the XGBoost Classification Metrics

View precision, recall, F1 score, and overall accuracy.

In [ ]:
print(
    classification_report(
        y_test,
        xgb_predictions,
        target_names=[
            "No Sepsis",
            "Sepsis"
        ],
        zero_division=0
    )
)

## Step 42: Save the XGBoost Results

Save the metrics so all three tree-based models can be compared.

In [ ]:
xgb_accuracy = accuracy_score(
    y_test,
    xgb_predictions
)

xgb_precision = precision_score(
    y_test,
    xgb_predictions,
    zero_division=0
)

xgb_recall = recall_score(
    y_test,
    xgb_predictions,
    zero_division=0
)

xgb_f1 = f1_score(
    y_test,
    xgb_predictions,
    zero_division=0
)

print(
    "Accuracy:",
    round(
        xgb_accuracy,
        3
    )
)

print(
    "Precision:",
    round(
        xgb_precision,
        3
    )
)

print(
    "Recall:",
    round(
        xgb_recall,
        3
    )
)

print(
    "F1:",
    round(
        xgb_f1,
        3
    )
)

## Step 43: View XGBoost Feature Importance

Show the 10 features XGBoost relied on most strongly.

In [ ]:
xgb_classifier = (
    xgboost_model
    .named_steps[
        "classifier"
    ]
)

xgb_importance = pd.Series(
    xgb_classifier.feature_importances_,
    index=MODEL_FEATURES
)

xgb_importance = (
    xgb_importance
    .sort_values(
        ascending=False
    )
    .head(10)
    .sort_values()
)

plt.figure(
    figsize=(9, 6)
)

plt.barh(
    xgb_importance.index,
    xgb_importance.values
)

plt.title(
    "XGBoost - Top 10 Feature Importances"
)

plt.xlabel(
    "Feature Importance"
)

plt.show()

# Compare the Models

## Step 44: Compare Decision Tree, Random Forest, and XGBoost

Compare Accuracy, Precision, Recall, and F1 score using the same patient-held-out test set.

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        dt_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "Precision": [
        dt_precision,
        rf_precision,
        xgb_precision
    ],
    "Recall": [
        dt_recall,
        rf_recall,
        xgb_recall
    ],
    "F1": [
        dt_f1,
        rf_f1,
        xgb_f1
    ]
})

results.round(3)

## Step 45: Graph the Model Comparison

Plot the four metrics so differences between the three models are easier to see.

In [ ]:
results_plot = (
    results
    .set_index(
        "Model"
    )
)

results_plot.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title(
    "PhysioNet Sepsis Prediction Model Comparison"
)

plt.xlabel(
    "Model"
)

plt.ylabel(
    "Score"
)

plt.ylim(
    0,
    1
)

plt.xticks(
    rotation=0
)

plt.legend(
    loc="lower right"
)

plt.show()

## Step 46: Compare False Negatives

For a sepsis prediction problem, false negatives are especially important to examine because they represent SepsisLabel-positive hours the model failed to identify.

In [ ]:
def false_negatives(
    y_true,
    y_pred
):
    return int(
        (
            (y_true == 1)
            & (y_pred == 0)
        )
        .sum()
    )

false_negative_results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "False Negatives": [
        false_negatives(
            y_test,
            dt_predictions
        ),
        false_negatives(
            y_test,
            rf_predictions
        ),
        false_negatives(
            y_test,
            xgb_predictions
        )
    ]
})

false_negative_results

## Step 47: Predict One Sepsis-Positive ICU Record

Select one positive test record, when available, and compare how all three models classify the exact same ICU hour.

In [ ]:
positive_test_rows = (
    y_test[
        y_test == 1
    ]
    .index
)

if len(positive_test_rows) > 0:
    example_index = positive_test_rows[0]
else:
    example_index = X_test.index[0]

example_patient = (
    X_test
    .loc[
        [example_index]
    ]
)

actual_label = int(
    y_test.loc[
        example_index
    ]
)

example_results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "XGBoost"
    ],
    "Prediction": [
        int(
            decision_tree
            .predict(
                example_patient
            )[0]
        ),
        int(
            random_forest
            .predict(
                example_patient
            )[0]
        ),
        int(
            xgboost_model
            .predict(
                example_patient
            )[0]
        )
    ],
    "Sepsis Probability": [
        float(
            decision_tree
            .predict_proba(
                example_patient
            )[0, 1]
        ),
        float(
            random_forest
            .predict_proba(
                example_patient
            )[0, 1]
        ),
        float(
            xgboost_model
            .predict_proba(
                example_patient
            )[0, 1]
        )
    ]
})

example_results["Prediction"] = (
    example_results[
        "Prediction"
    ]
    .map({
        0: "No Sepsis",
        1: "Sepsis"
    })
)

print(
    "Actual label:",
    (
        "Sepsis"
        if actual_label == 1
        else "No Sepsis"
    )
)

example_results

# Discussion

After running the notebook, compare the three models:

- Did Random Forest improve on the single Decision Tree?
- Did XGBoost improve every metric?
- Which model had the highest Recall?
- Which model produced the fewest false negatives?
- Did Accuracy tell the same story as Precision, Recall, and F1?
- Which clinical features were most important?
- Did the single-feature leakage screen reveal any suspicious shortcuts?
- Why did we exclude `patient_id`?
- Why did we split by patient instead of randomly splitting hourly rows?
- Why did we exclude `source_set` from the baseline model?
- Would you keep `ICULOS` in a research model? Why or why not?

## Important Research Lesson

Very high performance should always trigger a leakage review.

With longitudinal medical data, common leakage sources include:

- Giving the model the target directly
- Putting the same patient in both training and testing
- Using measurements recorded after the intended prediction time
- Using hospital/site identifiers that allow shortcut learning
- Using engineered variables that were calculated with future information

For a true research project, the prediction time and observation window should be defined before feature engineering begins.